# Streaming Agent
> Understand the streaming data models and patterns for real-time responses.

Anchor provides `StreamDelta`, `StreamResult`, and `StreamUsage` models
for tracking streaming LLM responses. This notebook explores these models
and shows common streaming patterns.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.models.streaming import StreamDelta, StreamResult, StreamUsage

## StreamDelta
Each text chunk from a streaming response is represented as a
`StreamDelta` with the text fragment and its position index.

In [ ]:
# Simulate a stream of deltas
deltas = [
    StreamDelta(text="Hello", index=0),
    StreamDelta(text=" there", index=1),
    StreamDelta(text="! How", index=2),
    StreamDelta(text=" can I", index=3),
    StreamDelta(text=" help?", index=4),
]

print("Simulated stream deltas:")
for d in deltas:
    print(f"  Delta(index={d.index}, text={d.text!r})")

## Accumulate Deltas into Full Text
In practice, you concatenate deltas as they arrive to build
the complete response.

In [ ]:
accumulated = ""
print("Streaming output:")
for d in deltas:
    accumulated += d.text
    # In a real app: print(d.text, end="", flush=True)

print(f"  '{accumulated}'")
print(f"\nTotal deltas: {len(deltas)}")
print(f"Final length: {len(accumulated)} chars")

## StreamUsage
`StreamUsage` tracks token consumption from a completed response,
including cache-related metrics.

In [ ]:
usage = StreamUsage(
    input_tokens=150,
    output_tokens=42,
    cache_creation_input_tokens=100,
    cache_read_input_tokens=50,
)

print("Token usage:")
print(f"  input_tokens:                {usage.input_tokens}")
print(f"  output_tokens:               {usage.output_tokens}")
print(f"  cache_creation_input_tokens:  {usage.cache_creation_input_tokens}")
print(f"  cache_read_input_tokens:      {usage.cache_read_input_tokens}")
print(f"  total tokens:                {usage.input_tokens + usage.output_tokens}")

## StreamResult
`StreamResult` is the final accumulated result from a completed
stream, combining text, usage, model info, and stop reason.

In [ ]:
result = StreamResult(
    text=accumulated,
    usage=usage,
    model="claude-haiku-4-5-20251001",
    stop_reason="end_turn",
)

print("StreamResult:")
print(f"  text:        {result.text!r}")
print(f"  model:       {result.model}")
print(f"  stop_reason: {result.stop_reason}")
print(f"  usage:       {result.usage.input_tokens} in / "
      f"{result.usage.output_tokens} out")

## Streaming Pattern: agent.chat()
`Agent.chat()` returns an iterator of text chunks. Each chunk
corresponds to a `StreamDelta.text` from the underlying API.

In [ ]:
# Conceptual streaming pattern
# (requires API key -- shown for illustration)
#
# for chunk in agent.chat("Explain streaming."):
#     print(chunk, end="", flush=True)
#     # Each chunk is a string (the text portion of a StreamDelta)
#
# After chat completes:
# result = agent.last_result  # ContextResult from the pipeline

print("agent.chat() streaming pattern:")
print("  1. Call agent.chat(message)")
print("  2. Iterate over yielded text chunks")
print("  3. Each chunk is a str (partial response)")
print("  4. Concatenate for full response")
print("  5. Access agent.last_result for pipeline metadata")

## Simulated Full Streaming Flow
Putting it all together: simulate a streaming response with
deltas, accumulation, and a final result.

In [ ]:
# Simulate a complete streaming flow
print("=== Simulated Streaming Flow ===")
print()

# Phase 1: Stream deltas
chunks = ["Anchor", " is a", " context", " engineering",
          " toolkit", " for", " building", " AI", " apps."]

print("Phase 1 - Streaming deltas:")
full_text = ""
for i, chunk in enumerate(chunks):
    delta = StreamDelta(text=chunk, index=i)
    full_text += delta.text
    print(f"  [{i}] '{chunk}'")

print(f"\nPhase 2 - Accumulated text:")
print(f"  '{full_text}'")

# Phase 3: Final result
final = StreamResult(
    text=full_text,
    usage=StreamUsage(input_tokens=85, output_tokens=len(chunks)),
    model="claude-haiku-4-5-20251001",
    stop_reason="end_turn",
)

print(f"\nPhase 3 - Final StreamResult:")
print(f"  model:       {final.model}")
print(f"  stop_reason: {final.stop_reason}")
print(f"  tokens:      {final.usage.input_tokens} in / "
      f"{final.usage.output_tokens} out")

## Async Streaming
`Agent.achat()` provides the same streaming interface for
async applications.

In [ ]:
# Async streaming pattern (requires async context)
#
# async for chunk in agent.achat("Explain async streaming."):
#     print(chunk, end="", flush=True)

print("Async pattern: agent.achat()")
print("  - Same interface as agent.chat()")
print("  - Uses 'async for' instead of 'for'")
print("  - Pipeline builds with pipeline.abuild()")
print("  - Retries use asyncio.sleep()")

## Key Takeaways
- `StreamDelta`: individual text chunk with position index
- `StreamUsage`: token counts including cache metrics
- `StreamResult`: final accumulated response (text + usage + metadata)
- `agent.chat()` yields `str` chunks (sync iterator)
- `agent.achat()` yields `str` chunks (async iterator)
- Access `agent.last_result` after chat for the `ContextResult`

**Back to:** [Agents README](README.md)